# ARC-v0.14 — FEVER Mechanism Audit

Post-hoc mechanism audit using sealed ARC-v0.13 FIT and untouched validation artifacts. No retrieval rerun, no test access, frozen regime threshold |H3|=0.002.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib, json, warnings
import numpy as np
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss
from scipy.stats import pearsonr, spearmanr

warnings.filterwarnings("ignore", category=FutureWarning)
SEED=20260816
EPS=0.002
BOOTSTRAP_REPS=1000
ARC_ROOT=Path("/content/drive/MyDrive/rag-pq-checkpoints/arc-v0")
V013_RUN=ARC_ROOT/"fever-boundary-external-replication-v013"/"20260817-140640"
V014_OUT=ARC_ROOT/"fever-mechanism-audit-v014"/datetime.now(timezone.utc).strftime("%Y%m%d-%H%M%S")
V014_OUT.mkdir(parents=True, exist_ok=False)
PROTOCOL_PATH=V013_RUN/"v013_fever_boundary_protocol.json"
assert V013_RUN.is_dir() and PROTOCOL_PATH.is_file()
protocol=json.loads(PROTOCOL_PATH.read_text(encoding="utf-8"))
assert protocol["test_access_allowed"] is False
assert np.isclose(float(protocol["regime_threshold_abs_slope"]), EPS)
assert protocol["boundary_grid_config_count"]==44
print("Source:",V013_RUN)
print("Output:",V014_OUT)
print("Protocol:",protocol["status"])
print("ARC-v0.14 PREFLIGHT — PASS")

In [ ]:
def sha256_file(path, chunk_size=16*1024*1024):
    h=hashlib.sha256()
    with open(path,"rb") as f:
        while True:
            block=f.read(chunk_size)
            if not block: break
            h.update(block)
    return h.hexdigest()

def slopes_from_trajectory(df):
    group_cols=["query_id","low","high","method","alpha","k","temperature","config_key"]
    rows=[]
    for keys,g in df.groupby(group_cols,dropna=False,sort=False):
        g=g.sort_values("iteration")
        x=g["iteration"].to_numpy(np.float64)
        row=dict(zip(group_cols,keys))
        for src,dst in [("query_divergence","H1_slope"),("candidate_increment","H2_slope"),("abs_utility_gap","H3_slope")]:
            row[dst]=float(np.polyfit(x,g[src].to_numpy(np.float64),1)[0])
        rows.append(row)
    return pd.DataFrame(rows)

def trajectory_features(df):
    gcols=["query_id","method","alpha","k","temperature","config_key"]
    rows=[]
    for keys,g in df.groupby(gcols,dropna=False,sort=False):
        g=g.sort_values("iteration")
        def at_round(col,t):
            hit=g.loc[g["iteration"]==t,col]
            return float(hit.iloc[0]) if len(hit) else np.nan
        row=dict(zip(gcols,keys))
        row.update({
            "initial_abs_utility_gap":at_round("abs_utility_gap",0),
            "t1_query_divergence":at_round("query_divergence",1),
            "t1_candidate_increment":at_round("candidate_increment",1),
            "t1_abs_utility_gap":at_round("abs_utility_gap",1),
            "max_query_divergence":float(g["query_divergence"].max()),
            "max_candidate_increment":float(g["candidate_increment"].max()),
            "final_abs_utility_gap":float(g["abs_utility_gap"].iloc[-1]),
        })
        rows.append(row)
    return pd.DataFrame(rows)

def add_regime(df):
    out=df.copy(); h3=out["H3_slope"].to_numpy(np.float64)
    out["is_amplifying"]=(h3>EPS).astype(int)
    out["is_reversal"]=(h3<-EPS).astype(int)
    out["is_stable_or_null"]=((h3<=EPS)&(h3>=-EPS)).astype(int)
    out["regime"]=np.select([h3>EPS,h3<-EPS],["amplifying","reversal"],default="stable_or_null")
    out["is_softmax"]=(out["method"]=="softmax").astype(float)
    out["temperature_numeric"]=pd.to_numeric(out["temperature"],errors="coerce").fillna(1.0)
    out["log_k"]=np.log(out["k"].astype(float))
    return out

In [ ]:
fit_files=sorted(V013_RUN.glob("fit-*.parquet"))
val_files=sorted(V013_RUN.glob("validation-*.parquet"))
print("FIT checkpoints:",len(fit_files))
print("VAL checkpoints:",len(val_files))
assert len(fit_files)==44 and len(val_files)==44
fit_traj=pd.concat([pd.read_parquet(p) for p in fit_files],ignore_index=True)
val_traj=pd.concat([pd.read_parquet(p) for p in val_files],ignore_index=True)
assert fit_traj["query_id"].nunique()==3350
assert val_traj["query_id"].nunique()==3316
print("FIT trajectory:",fit_traj.shape)
print("VAL trajectory:",val_traj.shape)
print("TRAJECTORY LOAD — PASS")

In [ ]:
join_cols=["query_id","method","alpha","k","temperature","config_key"]
fit=add_regime(slopes_from_trajectory(fit_traj).merge(trajectory_features(fit_traj),on=join_cols,how="left",validate="one_to_one"))
val=add_regime(slopes_from_trajectory(val_traj).merge(trajectory_features(val_traj),on=join_cols,how="left",validate="one_to_one"))
assert len(fit)==3350*44 and len(val)==3316*44
print("FIT:",fit.shape,"VAL:",val.shape)
display(fit["regime"].value_counts(normalize=True).rename("fit_fraction").to_frame())
display(val["regime"].value_counts(normalize=True).rename("validation_fraction").to_frame())
print("MECHANISM TABLE RECONSTRUCTION — PASS")

In [ ]:
def split_headline(df,split):
    return {"split":split,"n_rows":len(df),"n_queries":df["query_id"].nunique(),"mean_H3":float(df["H3_slope"].mean()),"stable_or_null":float(df["is_stable_or_null"].mean()),"amplifying":float(df["is_amplifying"].mean()),"reversal":float(df["is_reversal"].mean())}
headline=pd.DataFrame([split_headline(fit,"fit"),split_headline(val,"validation")])
display(headline)
headline.to_csv(V014_OUT/"v014_fit_validation_headline.csv",index=False)

In [ ]:
def config_risk(df,prefix):
    out=df.groupby(["config_key","method","alpha","k","temperature"],dropna=False,as_index=False).agg(amplifying_fraction=("is_amplifying","mean"),reversal_fraction=("is_reversal","mean"),mean_H3=("H3_slope","mean"))
    return out.rename(columns={"amplifying_fraction":f"{prefix}_amplifying_fraction","reversal_fraction":f"{prefix}_reversal_fraction","mean_H3":f"{prefix}_mean_H3"})
cfg_rep=config_risk(fit,"fit").merge(config_risk(val,"val"),on=["config_key","method","alpha","k","temperature"],how="inner",validate="one_to_one")
pearson_amp=pearsonr(cfg_rep["fit_amplifying_fraction"],cfg_rep["val_amplifying_fraction"])
spearman_amp=spearmanr(cfg_rep["fit_amplifying_fraction"],cfg_rep["val_amplifying_fraction"])
pearson_h3=pearsonr(cfg_rep["fit_mean_H3"],cfg_rep["val_mean_H3"])
print("Amplification Pearson r:",pearson_amp.statistic,"p=",pearson_amp.pvalue)
print("Amplification Spearman rho:",spearman_amp.statistic,"p=",spearman_amp.pvalue)
print("Mean-H3 Pearson r:",pearson_h3.statistic,"p=",pearson_h3.pvalue)
display(cfg_rep.sort_values("val_amplifying_fraction",ascending=False).head(15))
cfg_rep.to_csv(V014_OUT/"v014_config_risk_replication.csv",index=False)

In [ ]:
def alpha_summary(df,split):
    return df.groupby("alpha",as_index=False).agg(amplifying_fraction=("is_amplifying","mean"),stable_or_null_fraction=("is_stable_or_null","mean"),reversal_fraction=("is_reversal","mean"),mean_H3=("H3_slope","mean")).assign(split=split)
alpha_table=pd.concat([alpha_summary(fit,"fit"),alpha_summary(val,"validation")],ignore_index=True)
display(alpha_table)
for split,g in alpha_table.groupby("split"):
    amp=g.sort_values("alpha")["amplifying_fraction"].to_numpy()
    print(split,"amplification by alpha:",amp,"monotone:",bool(np.all(np.diff(amp)>=0)))
alpha_table.to_csv(V014_OUT/"v014_alpha_dose_response.csv",index=False)

In [ ]:
def query_susceptibility(df,split):
    q=df.groupby("query_id",as_index=False).agg(amplification_rate=("is_amplifying","mean"),reversal_rate=("is_reversal","mean"),mean_H3=("H3_slope","mean"),max_H3=("H3_slope","max"),initial_abs_utility_gap=("initial_abs_utility_gap","first"))
    q["split"]=split
    return q
q_fit=query_susceptibility(fit,"fit"); q_val=query_susceptibility(val,"validation")
query_summary=pd.DataFrame([{
"split":"fit","never_amplifying":float((q_fit.amplification_rate==0).mean()),"any_amplifying":float((q_fit.amplification_rate>0).mean()),"amp_ge_25pct_configs":float((q_fit.amplification_rate>=0.25).mean()),"amp_ge_50pct_configs":float((q_fit.amplification_rate>=0.50).mean())},{
"split":"validation","never_amplifying":float((q_val.amplification_rate==0).mean()),"any_amplifying":float((q_val.amplification_rate>0).mean()),"amp_ge_25pct_configs":float((q_val.amplification_rate>=0.25).mean()),"amp_ge_50pct_configs":float((q_val.amplification_rate>=0.50).mean())}])
display(query_summary)
pd.concat([q_fit,q_val],ignore_index=True).to_csv(V014_OUT/"v014_query_susceptibility.csv",index=False)
query_summary.to_csv(V014_OUT/"v014_query_susceptibility_headline.csv",index=False)

In [ ]:
PRE_FEEDBACK_FEATURES=["initial_abs_utility_gap","alpha","log_k","is_softmax","temperature_numeric"]
X_fit=fit[PRE_FEEDBACK_FEATURES].to_numpy(np.float64); y_fit=fit["is_amplifying"].to_numpy(int)
X_val=val[PRE_FEEDBACK_FEATURES].to_numpy(np.float64); y_val=val["is_amplifying"].to_numpy(int)
assert np.isfinite(X_fit).all() and np.isfinite(X_val).all()
clf=Pipeline([("scale",StandardScaler()),("model",LogisticRegression(C=1.0,max_iter=5000,class_weight="balanced",random_state=SEED))])
clf.fit(X_fit,y_fit)
p_fit=clf.predict_proba(X_fit)[:,1]; p_val=clf.predict_proba(X_val)[:,1]
metrics=pd.DataFrame([{
"split":"fit","roc_auc":roc_auc_score(y_fit,p_fit),"pr_auc":average_precision_score(y_fit,p_fit),"prevalence":y_fit.mean(),"brier":brier_score_loss(y_fit,p_fit)},{
"split":"validation","roc_auc":roc_auc_score(y_val,p_val),"pr_auc":average_precision_score(y_val,p_val),"prevalence":y_val.mean(),"brier":brier_score_loss(y_val,p_val)}])
coef=pd.DataFrame({"feature":PRE_FEEDBACK_FEATURES,"standardized_logistic_coefficient":clf.named_steps["model"].coef_[0]})
display(metrics); display(coef.reindex(coef.standardized_logistic_coefficient.abs().sort_values(ascending=False).index))
metrics.to_csv(V014_OUT/"v014_pre_feedback_prediction_metrics.csv",index=False)
coef.to_csv(V014_OUT/"v014_pre_feedback_prediction_coefficients.csv",index=False)

In [ ]:
val_pred=val[["query_id","is_amplifying"]].copy(); val_pred["probability"]=p_val
groups={qid:g for qid,g in val_pred.groupby("query_id",sort=False)}
qids=np.array(list(groups.keys()),dtype=object); rng=np.random.default_rng(SEED)
boot_auc=[]; boot_ap=[]
for _ in range(BOOTSTRAP_REPS):
    sampled=rng.choice(qids,size=len(qids),replace=True)
    yb=np.concatenate([groups[q]["is_amplifying"].to_numpy(int) for q in sampled])
    pb=np.concatenate([groups[q]["probability"].to_numpy(np.float64) for q in sampled])
    if np.unique(yb).size<2: continue
    boot_auc.append(roc_auc_score(yb,pb)); boot_ap.append(average_precision_score(yb,pb))
bootstrap_metrics=pd.DataFrame([{
"metric":"roc_auc","estimate":roc_auc_score(y_val,p_val),"ci_low":np.quantile(boot_auc,0.025),"ci_high":np.quantile(boot_auc,0.975),"bootstrap_reps":len(boot_auc)},{
"metric":"pr_auc","estimate":average_precision_score(y_val,p_val),"ci_low":np.quantile(boot_ap,0.025),"ci_high":np.quantile(boot_ap,0.975),"bootstrap_reps":len(boot_ap)}])
display(bootstrap_metrics)
bootstrap_metrics.to_csv(V014_OUT/"v014_validation_cluster_bootstrap_prediction.csv",index=False)

In [ ]:
MEDIATORS=["t1_query_divergence","t1_candidate_increment","t1_abs_utility_gap","max_query_divergence","max_candidate_increment"]
rows=[]; effects=[]
for split,df in [("fit",fit),("validation",val)]:
    for feature in MEDIATORS:
        for regime in ["amplifying","stable_or_null","reversal"]:
            x=df.loc[df.regime==regime,feature].to_numpy(np.float64)
            rows.append({"split":split,"feature":feature,"regime":regime,"n":len(x),"mean":float(np.nanmean(x)),"median":float(np.nanmedian(x)),"std":float(np.nanstd(x))})
        a=df.loc[df.regime=="amplifying",feature].to_numpy(np.float64); s=df.loc[df.regime=="stable_or_null",feature].to_numpy(np.float64)
        ma,ms=np.nanmean(a),np.nanmean(s); va,vs=np.nanvar(a,ddof=1),np.nanvar(s,ddof=1); pooled=np.sqrt((va+vs)/2)
        effects.append({"split":split,"feature":feature,"amplifying_mean":ma,"stable_mean":ms,"mean_difference":ma-ms,"standardized_mean_difference":(ma-ms)/pooled if pooled>0 else np.nan})
med=pd.DataFrame(rows); eff=pd.DataFrame(effects)
display(med); display(eff.sort_values(["split","standardized_mean_difference"],ascending=[True,False]))
med.to_csv(V014_OUT/"v014_mediator_regime_summary.csv",index=False)
eff.to_csv(V014_OUT/"v014_mediator_effect_sizes.csv",index=False)

In [ ]:
val_metrics_row=metrics.loc[metrics.split=="validation"].iloc[0]
report={
"status":"ARC_V014_FEVER_MECHANISM_AUDIT_COMPLETE",
"analysis_scope":"Post-hoc mechanism audit using sealed ARC-v0.13 FIT and untouched validation trajectory checkpoints. No retrieval rerun and no test access.",
"source_v013_protocol_sha256":sha256_file(PROTOCOL_PATH),
"source_v013_validation_report_sha256":sha256_file(V013_RUN/"v013_validation_continuation_report.json"),
"regime_threshold_abs_slope":EPS,
"threshold_status":"pre-existing frozen ARC-v0.13 regime threshold",
"fit_rows":int(len(fit)),"validation_rows":int(len(val)),
"fit_queries":int(fit.query_id.nunique()),"validation_queries":int(val.query_id.nunique()),
"config_count":44,
"config_amplification_pearson_r":float(pearson_amp.statistic),
"config_amplification_spearman_rho":float(spearman_amp.statistic),
"validation_pre_feedback_roc_auc":float(val_metrics_row.roc_auc),
"validation_pre_feedback_pr_auc":float(val_metrics_row.pr_auc),
"validation_amplification_prevalence":float(val_metrics_row.prevalence),
"predictive_feature_scope":PRE_FEEDBACK_FEATURES,
"mediator_scope":MEDIATORS,
"interpretation_constraints":[
"The pre-feedback predictive model is post-hoc and must not be described as a preregistered primary endpoint.",
"Round-1 and trajectory mediator quantities occur after feedback begins and must not be described as deployable pre-retrieval predictors.",
"The original q75 high-amplification target remains degenerate on FEVER."],
"retrieval_rerun":False,"test_accessed":False,
"completed_at_utc":datetime.now(timezone.utc).isoformat()}
REPORT_PATH=V014_OUT/"v014_mechanism_audit_report.json"
REPORT_PATH.write_text(json.dumps(report,indent=2,sort_keys=True),encoding="utf-8")
report_sha=sha256_file(REPORT_PATH)
(V014_OUT/"V014_REPORT_SHA256.txt").write_text(report_sha+"  "+REPORT_PATH.name+"\n",encoding="utf-8")
print("="*80)
print("ARC-v0.14 MECHANISM AUDIT — PASS")
print("Output:",V014_OUT)
print("Report SHA-256:",report_sha)
print("Validation ROC-AUC:",float(val_metrics_row.roc_auc))
print("Validation PR-AUC:",float(val_metrics_row.pr_auc))
print("Validation prevalence:",float(val_metrics_row.prevalence))
print("Config FIT↔VAL Spearman:",float(spearman_amp.statistic))
print("="*80)